# Step 6: group_size 调参

**目标**：理解 `group_size`（分组量化时每组共享一个 scale 的通道数）对精度与显存的影响——太小（32）scale 开销占比上升、压缩比下降；太大（256+）组内离群点影响扩大；**128 经验最优**。掌握 `group_size` 必须整除 `hidden_size`（否则 vLLM 加载报错）这一硬约束。

**对应 OUTLINE 课时**：3.6 group_size 调参（~30 分钟，偏实验）。


## 学完应能讲清（学完本节应能口头回答）
1. `group_size` 是什么？它和「per-channel 量化」「per-tensor 量化」是什么关系？（group=通道数→per-channel；group=全部→per-tensor；中间→分组）
2. 为什么 `group_size=128` 是经验最优？（32 以下 scale 开销吃掉压缩比；256+ 组内离群点撑爆 scale）
3. 为什么 `group_size` **必须整除 hidden_size**？不整除会怎样？（vLLM 加载报错，权重切分对不上）
4. 调 `group_size` 时「精度」与「显存/压缩比」是**反向**的两个目标——往小调（如 64）和往大调（如 256）分别换到什么、付出什么？提示：往小调→每组更精细（scale 多、精度↑）但 scale 数翻倍、压缩比略降、显存略涨；往大调→scale 更少、压缩比↑、省显存，但组内更可能混进离群点撑爆该组 scale、精度↓。所以「显存紧张」应往**大**调（省 scale），「精度优先」应往**小**调（更精细）——方向相反，别记反。
5. 给定 hidden_size=4096，哪些 group_size 合法？4096 本身（per-tensor 的退化）合不合法？

In [ ]:
%%capture
import pathlib, os, math
import torch
import torch.nn as nn
import ipytest
ipytest.autoconfig()
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from transformers import Qwen2Config, Qwen2ForCausalLM
from llmcompressor.modifiers.quantization import QuantizationModifier


In [ ]:
# Setup cell（双 env：模块根 = 含 scripts/ + steps/ 的 course/m3-tuning-eval/）。
import pathlib

def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts").is_dir() and (cand / "steps").is_dir():
            return cand
    raise RuntimeError("找不到模块根（含 scripts/ + steps/ 的目录）")

MODULE_ROOT      = _find_module_root(pathlib.Path.cwd())
MODEL_DIR        = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"
TINY_MODEL_DIR   = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"
OUT_ROOT         = MODULE_ROOT / "out"; OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("MODULE_ROOT =", MODULE_ROOT, "| 0.5B @", TINY_MODEL_DIR.exists())


## 原理：group_size 的三向权衡（OUTLINE 3.6）

权重量化按「多少个通道共享一个 scale」分档：

| 模式 | group_size | scale 数 | 精度 | 压缩比 |
|---|---|---|---|---|
per-tensor | =权重总数 | 1 | 差（一个 scale 管全部，离群点撑爆）| 最高 |
分组 | 128（经验最优）| hidden/128 | 好 | 高 |
分组 | 64 | hidden/64 | 更好（每组更精细）| 略低（scale 翻倍）|
分组 | 32 | hidden/32 | 更好 | 低（scale 开销吃压缩比）|
per-channel | 1（或=1 行）| 每行 1 | 最好 | 最低 |

**经验共识**（OUTLINE）：`group_size=128` 在常见 hidden_size（4096/8192 的整数因子）下平衡压缩比与精度。

**两个关键约束/易错点**：
- **必须整除 hidden_size**：vLLM/compressed-tensors 按组切分权重加载，不整除切分对不上 → 加载报错。所以合法 group_size 是 hidden_size 的**因子**。
- **不是越小越好**：32 以下，scale 的存储开销（每 group 一个 fp16 scale）占比上升，压缩比反而变差（OUTLINE 易错点）。


### 端到端：group_size 在调优里的位置

s1-s5 调的是「**哪些层**量化/回退」（层选择）；s6 调的是「**每层怎么量化**」（量化粒度）。两者正交：你可以对选定的层用不同 group_size。group_size 是 s7 评测对比的一个维度（同一模型 group=64 vs 128 的精度/显存差异）。


## 亲手摸一摸：不同 group_size 的 scale 数量 + 合法性

给定 hidden_size=4096（Qwen2.5-7B 的实际值），算各 group_size 的 scale 数量，并判断合法性（是否整除）。


In [ ]:
## 摸一摸：hidden_size=4096 下各 group_size 的 scale 数与合法性
hidden = 4096
candidates = [32, 64, 128, 256, 170, 4096]
print(f"hidden_size={hidden}：")
print(f"{'group_size':>10} {'scale数':>8} {'合法':>6} {'压缩比评级':>10}")
for gs in candidates:
    n_scales = hidden // gs if hidden % gs == 0 else None
    legal = (hidden % gs == 0)
    rate = '最优' if gs==128 else ('太小组网开销大' if gs<=32 else ('太组内离群' if gs>=256 else '可'))
    print(f"{gs:>10} {str(n_scales):>8} {str(legal):>6} {rate:>10}")
print("\n170 不整除 4096 → 非法（vLLM 会拒加载）；4096=per-tensor 退化（合法但精度差）")


## 本步填空

1. **`compare_group_sizes(hidden_size, group_sizes)`** —— 给定 hidden_size 和一组候选 group_size，返回每个的 (scale 数量、是否合法、scale 存储开销占比)。**为什么这么设计（填前先想）**：调参决策需要量化对比「精度代理（scale 数越多精度越好）」vs「开销（scale 占权重的比例）」，一键产出对比表支撑选型。
2. **`validate_group_size(group_size, hidden_size)`**（判断型）—— 校验 group_size 是否合法（正数、整除 hidden_size、≤hidden_size），不合法给明确错误信息。**为什么这么设计**：这是 vLLM 加载的硬约束——配置前先 validate，比让 vLLM 加载时报晦涩错误友好得多。


In [ ]:
def compare_group_sizes(hidden_size, group_sizes):
    """对每个候选 group_size，返回 dict：
      {group_size, num_scales, legal, scale_overhead_frac}
    - num_scales = hidden_size // group_size（仅 legal 时）。
    - legal = hidden_size % group_size == 0 且 1 <= group_size <= hidden_size。
    - scale_overhead_frac = num_scales / hidden_size（每个 scale 是 1 个 fp16，
      权重若 W4 则每 4 bit 一个 scale，开销 = scales_bits/weights_bits ≈ num_scales/hidden * 16/4）。
      这里返回裸 num_scales/hidden 作相对开销代理。

    为什么这么设计（填前先想）：调参要对比「scale 数（精度代理，多=精）」vs
    「scale 开销占比（少=压得狠）」。128 通常是这两者的拐点。
    """
    # TODO: 对 group_sizes 每个 gs：
    #   1) legal = isinstance(gs,int) and 1 <= gs <= hidden_size and hidden_size % gs == 0。
    #   2) num_scales = hidden_size // gs if legal else None。
    #   3) overhead = num_scales / hidden_size if legal else None。
    #   4) append dict。
    #   返回 list。
    raise NotImplementedError


In [ ]:
def validate_group_size(group_size, hidden_size):
    """判断型：校验 group_size 合法性。
    合法条件：正整数、<= hidden_size、且整除 hidden_size。
    合法返回 True；非法 raise ValueError（带明确原因）。

    为什么这么设计（填前先想）：group_size 必须整除 hidden_size 是 vLLM 加载的硬约束
    （按组切权重）。配置前主动 validate，比让 vLLM 在加载时报晦涩张量形状错误友好得多。
    """
    # TODO: 依次校验，非法就 raise ValueError(明确原因)：
    #   1) isinstance(group_size, int) and group_size >= 1，否则 "group_size 必须是 >=1 的整数"。
    #   2) group_size <= hidden_size，否则 "group_size 不能大于 hidden_size"。
    #   3) hidden_size % group_size == 0，否则 f"group_size={group_size} 不整除 hidden_size={hidden_size}"。
    #   全过返回 True。
    raise NotImplementedError


In [ ]:
def test_compare_group_sizes_legal_and_illegal():
    out = compare_group_sizes(4096, [32, 64, 128, 256, 170, 4096])
    by_gs = {d["group_size"]: d for d in out}
    assert by_gs[128]["legal"] is True and by_gs[128]["num_scales"] == 32
    assert by_gs[170]["legal"] is False and by_gs[170]["num_scales"] is None
    assert by_gs[4096]["legal"] is True and by_gs[4096]["num_scales"] == 1  # per-tensor 退化
    # scale 数随 group_size 减小而增多（精度代理）
    assert by_gs[64]["num_scales"] > by_gs[128]["num_scales"] > by_gs[256]["num_scales"]

def test_validate_group_size_legal():
    assert validate_group_size(128, 4096) is True
    assert validate_group_size(1, 4096) is True      # per-channel 极端
    assert validate_group_size(4096, 4096) is True   # per-tensor 退化

def test_validate_group_size_illegal_not_divisor():
    import pytest
    with pytest.raises(ValueError, match="不整除"):
        validate_group_size(170, 4096)

def test_validate_group_size_illegal_too_big():
    import pytest
    with pytest.raises(ValueError, match="不能大于"):
        validate_group_size(8192, 4096)

def test_validate_group_size_illegal_nonpositive():
    import pytest
    with pytest.raises(ValueError, match=">=1"):
        validate_group_size(0, 4096)

# L1 必过守卫：ipytest.run 返回 pytest 退出码；非 0（有测试失败）→ 抛异常让 nbconvert 真挂。
# （用 ipytest.run() 而非 %%ipytest magic：magic 吞掉失败、exit_code 属性在本版不可靠。）
_ec = ipytest.run("-qq")
assert _ec == 0, f"L1 测试未全过（exit_code={_ec}），见上方 pytest 输出。"

## L2（tiny，CPU）：group_size 对比表 + 精度代理曲线

对 tiny 模型的 hidden_size 跑 group_size 对比，画「scale 数（精度代理）vs 开销占比」曲线——直观看到 128 在中间的平衡点。


In [ ]:
## L2：tiny hidden_size=64 的 group_size 对比
hidden = 64  # tiny 模型
candidates = [1, 2, 4, 8, 16, 32, 64]
rows = compare_group_sizes(hidden, candidates)
print(f"tiny hidden_size={hidden}：")
print(f"{'group':>6} {'scales':>7} {'legal':>6} {'overhead':>9}")
for r in rows:
    oh = f"{r['scale_overhead_frac']:.3f}" if r['scale_overhead_frac'] is not None else 'N/A'
    print(f"{r['group_size']:>6} {str(r['num_scales']):>7} {str(r['legal']):>6} {oh:>9}")

legal = [r for r in rows if r['legal']]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot([r['group_size'] for r in legal], [r['num_scales'] for r in legal], 'o-')
ax1.set_xlabel('group_size'); ax1.set_ylabel('Num scales (precision proxy)'); ax1.set_title('scale 数')
ax2.plot([r['group_size'] for r in legal], [r['scale_overhead_frac'] for r in legal], 's-')
ax2.set_xlabel('group_size'); ax2.set_ylabel('Overhead ratio (lower better)'); ax2.set_title('scale 开销占比')
fig.tight_layout(); plt.show()
assert all(r['legal'] for r in legal)
print("\nL2 通过：group_size 对比表 + 合法性校验逻辑正确（128 思想：精度代理与开销的平衡点）。")


## L3（H200，GPU 守卫）：真 7B group_size 精度/磁盘对比

在 7B 上跑 group_size=64 vs 128 vs 256 三档 W8A8 量化，对比 PPL（精度）与产物磁盘占用——实证「128 平衡、64 更精但更大、256 更小但精度降」。本节偏实验，L3 是核心证据。


In [ ]:
import torch, os
def run_l3_group_size():
    from llmcompressor.modifiers.transform.smoothquant import SmoothQuantModifier
    from datasets import load_dataset
    calib = load_dataset("wikitext", "wikitext-2-raw-v1", split="train").shuffle(seed=0)["text"][:64]
    import json
    hidden = json.loads(open(MODEL_DIR / 'config.json').read())['hidden_size']
    results = []
    for gs in [64, 128, 256]:
        validate_group_size(gs, hidden)
        from llmcompressor.modifiers.quantization import QuantizationModifier as QM
        recipe = [QM(targets="Linear", scheme="W4A16", ignore=["lm_head"],
                     observer="minmax", observer_kwargs={"group_size": gs})]
        out = OUT_ROOT / f"s6_group{gs}"
        oneshot(model=str(MODEL_DIR), tokenizer=str(MODEL_DIR), recipe=recipe,
                dataset=calib, num_calibration_samples=64, output_dir=str(out))
        disk = sum(f.stat().st_size for f in out.glob('**/*') if f.is_file()) / 1e9
        results.append({"group_size": gs, "disk_gb": round(disk, 2)})
        print(f"  group={gs} -> 磁盘 {disk:.2f} GB")
    json.dump(results, open(OUT_ROOT / "s6_group_compare.json", "w"), indent=2)

if torch.cuda.is_available() and not os.environ.get("SKIP_L3"):
    run_l3_group_size()
else:
    print("跳过 L3：无 GPU 或 SKIP_L3=1（CPU/CI 只验 L1+L2 对比/校验逻辑）")


## 产物检查


In [ ]:
import json
p = OUT_ROOT / "s6_group_compare.json"
if p.exists():
    for r in json.loads(p.read_text()):
        print(f"  group={r['group_size']:>4}  磁盘={r['disk_gb']:.2f} GB")
else:
    print(f"{p} 不存在（L3 未跑或被 SKIP_L3 跳过）")
